In [ ]:
import logging

import numpy as np

from atmodeller import (
    ChemicalSpecies,
    EquilibriumModel,
    Planet,
    SpeciesNetwork,
    debug_logger,
    earth_oceans_to_hydrogen_mass,
)
from atmodeller.eos import get_eos_models
from atmodeller.solubility import get_solubility_models
from atmodeller.thermodata import IronWustiteBuffer

logger = debug_logger()
logger.setLevel(logging.INFO)

# For more output use DEBUG
# logger.setLevel(logging.DEBUG)

In [35]:
pwd

'/Users/lukav/Documents/GitHub/atmodeller/notebooks'

# Basic Usage

This notebook is available at `notebooks/basic_usage.ipynb` and is easiest to obtain by downloading the source code.

## Species and thermodynamic data

The species available in *Atmodeller* can be found in the `thermodata` subpackage, where the prefix of the dictionary key denotes the chemical formula in *Hill notation* and the suffix describes the *states of aggregation* in accordance with the JANAF convention.

In [22]:
# Get all available species
available_species = SpeciesNetwork.available_species()
logger.info("Available species = %s", available_species)

# To create a gas species, for example CO2, where the state of aggregation defaults to 'g':
CO2_g = ChemicalSpecies.create_gas("CO2")

# The unique name that Atmodeller assigns combines the Hill notation and the state of aggregation
logger.info("Species name = %s", CO2_g.data.name)

# Compute the Gibbs energy relative to RT at 2000 K
temperature = 2000.0
gibbs = CO2_g.data.get_gibbs_over_RT(temperature, None)
logger.info("Gibbs/RT = %s", gibbs)

# Compute the composition
composition = CO2_g.data.composition
logger.info("Composition = %s", composition)

# Access more thermodynamic data
heat_capacity = CO2_g.data.thermo.cp(temperature)
logger.info("Heat capacity = %s", heat_capacity)

# Etc., other methods are available to compute other quantities

[18:25:07 - atmodeller                     - INFO     ] - Available species = ('Al_g', 'AlO_g', 'AlO2_g', 'Al2_g', 'Al2O_g', 'Al2O2_g', 'Al2O3_g', 'Ar_g', 'C_g', 'CN_g', 'CO_g', 'COS_g', 'CO2_g', 'C2H_g', 'C2N_g', 'C3_g', 'C4N2_g', 'CHN_g', 'CH2_g', 'CH3_g', 'CH4_g', 'C2H2_g', 'C2H3_g', 'C2H3N_g', 'C2H4_g', 'C2H5_g', 'C2H6_g', 'Ca_g', 'CaO_g', 'Ca2_g', 'Cl2_g', 'ClH_g', 'Cr_g', 'CrO_g', 'CrO2_g', 'CrO3_g', 'Fe_g', 'FeO_g', 'H_g', 'HO_g', 'HS_g', 'H2_g', 'H2O_g', 'H2O4S_g', 'H2S_g', 'H3N_g', 'H4Si_g', 'He_g', 'K_g', 'KO_g', 'K2_g', 'K2O_g', 'K2O2_g', 'Kr_g', 'Mg_g', 'MgO_g', 'N_g', 'NH_g', 'NH2_g', 'NO_g', 'N2_g', 'Na_g', 'NaO_g', 'Na2_g', 'Na2O_g', 'Na2O2_g', 'Ne_g', 'O_g', 'OS_g', 'OSi_g', 'OTi_g', 'O2_g', 'O2S_g', 'O2Si_g', 'O2Ti_g', 'O3_g', 'O3S_g', 'S2_g', 'Si_g', 'Ti_g', 'Xe_g', 'Al2O3_cd', 'C_cd', 'CSi_cd', 'CaO_cd', 'ClH4N_cd', 'Fe_cd', 'FeO_cd', 'Fe3O4_cd', 'H2O_cd', 'H2O4S_cd', 'MgO_cd', 'MgO3Si_cd', 'Mg2O4Si_cd', 'N4Si3_cd', 'O2Si_cd', 'S_cd', 'Si_cd')
[18:25:07 - atmodeller 

## Solubility

Solubility laws are available in the `solubility` subpackage.

In [23]:
solubility_models = get_solubility_models()
logger.info("Solubility models = %s", solubility_models.keys())

CO2_basalt = solubility_models["CO2_basalt_dixon95"]
# Compute the concentration at fCO2=0.5 bar, 1300 K, and 1 bar
# Note that fugacity is the first argument and others are keyword only
concentration = CO2_basalt.concentration(0.5, temperature=1300, pressure=1)
logger.info("Concentration (ppmw) = %s", concentration)

[18:25:07 - atmodeller                     - INFO     ] - Solubility models = dict_keys(['Ar_basalt_jambon86', 'CH4_basalt_ardia13', 'CO2_basalt_dixon95', 'CO_basalt_armstrong15', 'CO_basalt_yoshioka19', 'CO_rhyolite_yoshioka19', 'Cl2_ano_dio_for_thomas21', 'Cl2_basalt_thomas21', 'H2O_ano_dio_newcombe17', 'H2O_basalt_dixon95', 'H2O_basalt_mitchell17', 'H2O_lunar_glass_newcombe17', 'H2O_peridotite_sossi23', 'H2_andesite_hirschmann12', 'H2_basalt_hirschmann12', 'H2_chachan18', 'H2_kite19', 'H2_silicic_melts_gaillard03', 'He_basalt_jambon86', 'Kr_basalt_jambon86', 'N2_basalt_bernadou21', 'N2_basalt_dasgupta22', 'N2_basalt_libourel03', 'Ne_basalt_jambon86', 'S2_andesite_boulliung23', 'S2_basalt_boulliung23', 'S2_sulfate_andesite_boulliung23', 'S2_sulfate_basalt_boulliung23', 'S2_sulfate_trachybasalt_boulliung23', 'S2_sulfide_andesite_boulliung23', 'S2_sulfide_basalt_boulliung23', 'S2_sulfide_trachybasalt_boulliung23', 'S2_trachybasalt_boulliung23', 'Xe_basalt_jambon86', 'NO_SOLUBILITY'])
[

In [24]:
N2_basalt = solubility_models["N2_basalt_libourel03"]
# Compute the concentration at fCO2=0.5 bar, 1300 K, and 1 bar
# Note that fugacity is the first argument and others are keyword only
concentration = N2_basalt.concentration(0.20, temperature=1698.15, pressure=1, fO2=10**-16.2)
logger.info("Concentration (ppmw) = %s", concentration)

[18:25:07 - atmodeller                     - INFO     ] - Concentration (ppmw) = 377.1406984833259


In [25]:
N2_basalt_dasgupta = solubility_models["N2_basalt_dasgupta22"]
# Compute the concentration at fCO2=0.5 bar, 1300 K, and 1 bar
# Note that fugacity is the first argument and others are keyword only
concentration = N2_basalt_dasgupta.concentration(
    1550, temperature=1773.15, pressure=1708.7, fO2=1.8e-13
)
logger.info("Concentration (ppmw) = %s", concentration)

[18:25:07 - atmodeller                     - INFO     ] - Concentration (ppmw) = 1002.1074478063852


## Real gas EOS

Real gas equations of state are available in the `eos` subpackage.

In [26]:
# Get all available EOS models
eos_models = get_eos_models()
logger.info("EOS models = %s", eos_models.keys())

# Get a CH4 model
CH4_eos_model = eos_models["CH4_beattie_holley58"]
# Compute the fugacity at 800 K and 100 bar
fugacity = CH4_eos_model.fugacity(800, 100)
logger.info("Fugacity = %s bar", fugacity)
# Compute the compressibility factor at the same conditions
compressibility = CH4_eos_model.compressibility_factor(800, 100)
logger.info("Compressibility factor = %s", compressibility)
# Etc., other methods are available to compute other quantities

[18:25:07 - atmodeller                     - INFO     ] - EOS models = dict_keys(['H2_chabrier21', 'H2_He_Y0275_chabrier21', 'H2_He_Y0292_chabrier21', 'H2_He_Y0297_chabrier21', 'He_chabrier21', 'CH4_beattie_holley58', 'CO2_beattie_holley58', 'H2_beattie_holley58', 'He_beattie_holley58', 'N2_beattie_holley58', 'NH3_beattie_holley58', 'O2_beattie_holley58', 'CH4_cork_cs_holland91', 'CO_cork_cs_holland91', 'CO2_cork_holland91', 'CO2_cork_holland98', 'CO2_cork_cs_holland91', 'H2_cork_cs_holland91', 'H2O_cork_holland91', 'H2O_cork_holland98', 'H2S_cork_cs_holland11', 'N2_cork_cs_holland91', 'S2_cork_cs_holland11', 'OSi_rk49_connolly16', 'H4Si_rk49_reid87', 'CHN_rk49_reid87', 'H3N_rk49_reid87', 'Ar_cs_saxena87', 'CH4_cs_shi92', 'CO_cs_shi92', 'CO2_cs_shi92', 'COS_cs_shi92', 'H2_shi92', 'H2S_shi92', 'N2_cs_saxena87', 'O2_cs_shi92', 'S2_cs_shi92', 'SO2_shi92', 'H2_vdw_lide05', 'He_vdw_lide05', 'N2_vdw_lide05', 'H4Si_vdw_lide05', 'H2O_vdw_lide05', 'CH4_vdw_lide05', 'H3N_vdw_lide05', 'CHN_vdw_li

We can also use broadcasting to perform multiple evaluations at once, for example to compute a grid of fugacities:

In [27]:
# Define the temperature (K) and pressure (bar) grid
temperature = np.array([1000, 1600])
pressure = np.array([1, 10, 100])

temperature_broadcasted = temperature[:, None]
pressure_broadcasted = pressure[None, :]

# Get a CH4 model
CH4_eos_model = eos_models["CH4_cork_cs_holland91"]
# Compute the fugacity
fugacity = CH4_eos_model.fugacity(temperature_broadcasted, pressure_broadcasted)
logger.info("Fugacity = %s bar", fugacity)
# Compute the compressibility factor at the same conditions
compressibility = CH4_eos_model.compressibility_factor(
    temperature_broadcasted, pressure_broadcasted
)
logger.info("Compressibility factor = %s", compressibility)
# Etc., other methods are available to compute other quantities

[18:25:07 - atmodeller                     - INFO     ] - Fugacity = [[  1.                 10.036513019492777 104.04907255121003 ]
 [  1.                 10.027930271415434 103.07696646185612 ]] bar
[18:25:07 - atmodeller                     - INFO     ] - Compressibility factor = [[1.000406600334726 1.004038918021219 1.039882083041007]
 [1.000310800338152 1.003092414833786 1.030301664157559]]


## Model with mass constraints

A common scenario is to calculate how volatiles partition between a magma ocean and an atmosphere when the total elemental abundances are constrained. `Planet()` defaults to a molten Earth, but the planetary parameters can be changed using input arguments.

In [34]:
solubility_models = get_solubility_models()

H2_g = ChemicalSpecies.create_gas("H2")
H2O_g = ChemicalSpecies.create_gas("H2O", solubility=solubility_models["H2O_peridotite_sossi23"])
O2_g = ChemicalSpecies.create_gas("O2")

# This is one way of defining the species collection, and this approach is preferred if you want to
# specify species with solubility laws, as defined for H2O_g above.
species = SpeciesNetwork((H2_g, H2O_g, O2_g))

# Planet has input arguments that you can change. See the class documentation.
planet = Planet()
model = EquilibriumModel(species)

oceans = 1
h_kg = earth_oceans_to_hydrogen_mass(oceans)
o_kg = 6.25774e20
mass_constraints = {"H": h_kg, "O": o_kg}

model.solve(state=planet, mass_constraints=mass_constraints)
output = model.output

# Quick look at the solution
solution = output.quick_look()
logger.info("solution = %s", solution)

# Get complete solution as a dictionary
# solution_asdict = output.asdict()
# logger.info(solution_asdict)

# Get the complete solution as dataframes
# solution_dataframes = output.to_dataframes()

# Write the complete solution to Excel
# output.to_excel("example_mass_constraints")

[18:27:41 - atmodeller.classes             - INFO     ] - species_network = ('H2_g: IdealGas, NoSolubility', 'H2O_g: IdealGas, SolubilityPowerLaw', 'O2_g: IdealGas, NoSolubility')
[18:27:41 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[18:27:41 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g'}
[18:27:44 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[18:27:44 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[18:27:44 - atmodeller.classes             - INFO     ] - Solver steps (max) = 35
[18:27:44 - atmodeller                     - INFO     ] - solution = {'H2_g': array(15.168753777766597), 'H2_g_activity': array(15.16875377776659), 'H2O_g': array(0.066407208350572), 'H2O_g_activity': array(0.066407208350572), 'O2_g': array(1.591104671170275e-12), 'O2

log_number_moles = [50. 50. 50.]
log_stability = [-30. -30. -30.]
log_number_moles = [42.77130320395778  48.432447681268805 42.24251625970427 ]
log_stability = [-30. -30. -30.]
log_number_moles = [  80.                 65.44423094452799 -200.              ]
log_stability = [-30. -30. -30.]
log_number_moles = [79.00000000000068  64.43515799627613   3.866858038965574]
log_stability = [-30. -30. -30.]
log_number_moles = [78.00000000000253  63.4202217559475    3.836985614182729]
log_stability = [-30. -30. -30.]
log_number_moles = [77.00000000000756  62.395658459727294  3.787859112099205]
log_stability = [-30. -30. -30.]
log_number_moles = [76.00000000002126  61.355335202097145  3.707212741120047]
log_stability = [-30. -30. -30.]
log_number_moles = [75.00000000005849  60.28935054141529   3.575243645339877]
log_stability = [-30. -30. -30.]
log_number_moles = [74.00000000015969  59.18200312300564   3.360549149017435]
log_stability = [-30. -30. -30.]
log_number_moles = [73.00000000043475  58.0

## Model with fO2 constraint

Another common scenario is to calculate how volatiles partition between a magma ocean and an atmosphere when fO2 is fixed relative to a buffer and the total elemental abundances are constrained.

In [29]:
H2O_g = ChemicalSpecies.create_gas("H2O", solubility=solubility_models["H2O_peridotite_sossi23"])
H2_g = ChemicalSpecies.create_gas("H2")
O2_g = ChemicalSpecies.create_gas("O2")

species = SpeciesNetwork((H2O_g, H2_g, O2_g))

planet = Planet()
model = EquilibriumModel(species)

oceans = 1
h_kg = earth_oceans_to_hydrogen_mass(oceans)
mass_constraints = {"H": h_kg}

# Use the Iron Wustite buffer.  The "-1" argument is the log10 shift relative to the buffer.
fugacity_constraints = {"O2_g": IronWustiteBuffer(-1)}

model.solve(
    state=planet, fugacity_constraints=fugacity_constraints, mass_constraints=mass_constraints
)
output = model.output

# Quick look at the solution
solution = output.quick_look()
logger.info("solution = %s", solution)

[18:25:10 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility')
[18:25:10 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[18:25:10 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g'}
[18:25:14 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[18:25:14 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[18:25:14 - atmodeller.classes             - INFO     ] - Solver steps (max) = 10
[18:25:14 - atmodeller                     - INFO     ] - solution = {'H2O_g': array(0.252820417697318), 'H2O_g_activity': array(0.252820417697317), 'H2_g': array(0.774830277331649), 'H2_g_activity': array(0.774830277331647), 'O2_g': array(8.838513516896055e-09), 'O2_

log_number_moles = [50. 50. 50.]
log_stability = [-30. -30. -30.]
log_number_moles = [53.87676409604763 54.99672867920718 34.4159864401655 ]
log_stability = [-30. -30. -30.]
log_number_moles = [52.35310585904806  53.47307044220761  29.673942585788765]
log_stability = [-30. -30. -30.]
log_number_moles = [50.717626341255404 51.837590924414954 29.673942576946633]
log_stability = [-30. -30. -30.]
log_number_moles = [49.10891726151814 50.22888184467768 29.67394257530937]
log_stability = [-30. -30. -30.]
log_number_moles = [47.78445798033714 48.90442256349669 29.67394256827233]
log_stability = [-30. -30. -30.]
log_number_moles = [47.038693315566064 48.158657898725615 29.673942551718152]
log_stability = [-30. -30. -30.]
log_number_moles = [46.85250210648879  47.97246668964834  29.673942540636993]
log_stability = [-30. -30. -30.]
log_number_moles = [46.843036804862464 47.963001388022015 29.673942539712755]
log_stability = [-30. -30. -30.]
log_number_moles = [46.843013814715924 47.9629783978754

## Defining a network of species

In the above example, a species network was created by first creating the species and then aggregating the species into a `SpeciesNetwork`. This approach allows for complete generality since each species can also have its own solubility law assigned. However, if you want to create a simple gas network that assumes ideality and no solubility, you can also use the following formulation, where the *state of aggregation* must be specified after the formula (and separated by an underscore) for all species:

In [30]:
species_no_solubility = SpeciesNetwork.create(("H2_g", "H2O_g", "O2_g"))
logger.info("species_no_solubility = %s", species_no_solubility)

[18:25:14 - atmodeller                     - INFO     ] - species_no_solubility = ('H2_g: IdealGas, NoSolubility', 'H2O_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility')


## Batch calculation

For a batch calculation you can provide arrays to the planet or constraints. All arrays must have the same size because for a batch calculation the array values are aligned by position. Single values will automatically be broadcasted to the maximum array size.

In [31]:
solubility_models = get_solubility_models()

H2_g = ChemicalSpecies.create_gas("H2")
H2O_g = ChemicalSpecies.create_gas("H2O", solubility=solubility_models["H2O_peridotite_sossi23"])
O2_g = ChemicalSpecies.create_gas("O2")

species = SpeciesNetwork((H2_g, H2O_g, O2_g))

# Batch temperature and radius, where the entries correspond by position. You could also choose
# to leave one or both as scalars.
surface_temperature = np.array([2000, 2000, 1500, 1500])
earth_radius = 6371000  # m
surface_radius = earth_radius * np.array([1.5, 3, 1.5, 3])

planet = Planet(temperature=surface_temperature, surface_radius=surface_radius)
model = EquilibriumModel(species)

oceans = 1
h_kg = earth_oceans_to_hydrogen_mass(oceans)
o_kg = 6.25774e20
scale_factor = 5
mass_constraints = {
    # We can also batch constraints, as long as we also have a total of 4 entries
    "H": np.array([h_kg, h_kg, h_kg * scale_factor, h_kg * scale_factor]),
    "O": np.array([o_kg, o_kg * scale_factor, o_kg, o_kg * scale_factor]),
}

# Initial solution guess number of moles
initial_log_number_moles = 50

model.solve(
    state=planet,
    initial_log_number_moles=initial_log_number_moles,
    mass_constraints=mass_constraints,
)
output = model.output

# Quick look at the solution
solution = output.quick_look()
logger.info("Quick look = %s", solution)

# Get complete solution as a dictionary
# solution_asdict = output.asdict()
# logger.info(solution_asdict)

# Write the complete solution to Excel
# output.to_excel("example_batch")

[18:25:14 - atmodeller.classes             - INFO     ] - species_network = ('H2_g: IdealGas, NoSolubility', 'H2O_g: IdealGas, SolubilityPowerLaw', 'O2_g: IdealGas, NoSolubility')
[18:25:14 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[18:25:14 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g'}
[18:25:17 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 4 (100.00%) successful model(s)
[18:25:17 - atmodeller.classes             - INFO     ] - Multistart summary: 4 (100.00%) models(s) required 1 attempt(s)
[18:25:17 - atmodeller.classes             - INFO     ] - Solver steps (max) = 35
[18:25:17 - atmodeller                     - INFO     ] - Quick look = {'H2_g': array([3.332489632554158e+00, 3.261151703408882e-05,
       2.697322698724040e+01, 2.614487702886700e+00]), 'H2_g_activity': array([3.332489632554153e+00, 3.261151703408892e-05,
       2.6

log_number_moles = [50. 50. 50.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_number_moles = [50. 50. 50.]
log_number_moles = [50. 50. 50.]
log_number_moles = [50. 50. 50.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_number_moles = [43.931242596443035 48.985843846762904 42.822485955946824]
log_number_moles = [35.35486751452025 47.19551440625482 53.62187832636054]
log_number_moles = [66.48379483437415  61.54386649511278  24.438640794438562]
log_number_moles = [59.880272813048286 62.082980770831064 36.224459695150614]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_number_moles = [  80.                 59.48645792925601 -200.              ]
log_number_moles = [42.49196585761187  51.640531527035826 52.927816039525155]
log_

## Monte Carlo

Exploring atmospheric compositions in a Monte Carlo model can be achieved with a batch 
calculation over a range of parameters. Note that in this case the same initial solution is used 
for all cases.

In [32]:
solubility_models = get_solubility_models()

H2_g = ChemicalSpecies.create_gas("H2")
H2O_g = ChemicalSpecies.create_gas("H2O", solubility=solubility_models["H2O_peridotite_sossi23"])
O2_g = ChemicalSpecies.create_gas("O2")

species = SpeciesNetwork((H2_g, H2O_g, O2_g))
planet = Planet()
model = EquilibriumModel(species)

number_of_realisations = 1000
log10_number_oceans = np.random.uniform(0, 3, number_of_realisations)
number_oceans = 10**log10_number_oceans
fO2_min = -3
fO2_max = 3
fO2_log10_shifts = np.random.uniform(fO2_min, fO2_max, number_of_realisations)

oceans = 1
h_kg = earth_oceans_to_hydrogen_mass(number_oceans)
mass_constraints = {"H": h_kg}
fugacity_constraints = {O2_g.name: IronWustiteBuffer(fO2_log10_shifts)}

# Initial solution guess number of moles
initial_log_number_moles = 50 * np.ones(len(species))

model.solve(
    state=planet,
    initial_log_number_moles=initial_log_number_moles,
    mass_constraints=mass_constraints,
    fugacity_constraints=fugacity_constraints,
)
output = model.output

# Quick look at the solution
# solution = output.quick_look()
# logger.info("solution = %s", solution)

# Get complete solution as a dictionary
# solution_asdict = output.asdict()
# logger.info(solution_asdict)

# Write the complete solution to Excel
# output.to_excel("example_monte_carlo")

[18:25:17 - atmodeller.classes             - INFO     ] - species_network = ('H2_g: IdealGas, NoSolubility', 'H2O_g: IdealGas, SolubilityPowerLaw', 'O2_g: IdealGas, NoSolubility')
[18:25:17 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[18:25:17 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g'}


log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stabil

[18:25:58 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1000 (100.00%) successful model(s)
[18:25:58 - atmodeller.classes             - INFO     ] - Multistart summary: 1000 (100.00%) models(s) required 1 attempt(s)
[18:25:58 - atmodeller.classes             - INFO     ] - Solver steps (max) = 30


log_number_moles = [57.8874750096787   57.9621580586921   31.513450342937592]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_number_moles = [53.49833217823019 52.84139625090069 30.37169440063909]
log_number_moles = [57.850868259034804 55.90290321098494  28.412329196216803]
log_number_moles = [57.27119322391286 57.12630536030782 31.16318172995824]
log_number_moles = [52.85227927491094  50.429169329375775 27.65181833381777 ]
log_number_moles = [51.530008889291    48.365210266331    26.388157195637206]
log_number_moles = [54.37849510791746 54.56882109644503 31.7010658262524 ]
log_number_moles = [53.65873894415341  53.394941058453625 30.976510444593465]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. -30. -30.]
log_stability = [-30. 